# Exploratory Data Analysis 

## This notebook performs an exploratory analysis of the employment program data. 
It examines selection into treatment, tests baseline imbalances between participants and non-participants, and explores heterogeneous employment outcomes across risk groups. In particular, it investigates whether outcome gaps between private (OPP) and public (CVE) providers vary with ex-ante risk of long-term unemployment, providing descriptive evidence consistent with the “parking” hypothesis — namely, that performance-based incentives may lead private providers to allocate effort toward easier-to-place individuals. 

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

In [42]:
# Load the dataset
file_path = "data/dataPrivatePublic.dta"
df = pd.read_stata(file_path)
print(f"Dataset loaded with {df.shape[0]} observations.")

Dataset loaded with 43977 observations.


In [43]:
# Check dimensions (rows, columns)
print(f"Shape:{df.shape}")
# List columns names
print(f"Columns:{df.columns.values}")
# Summary of data type
df.info()

Shape:(43977, 83)
Columns:['ale' 'mois_saisie_occ' 'acceptationCVE' 'acceptationOPP' 'nregion'
 'ndem' 'sexe' 'nenf' 'nation' 'motins' 'temps' 'exper' 'age' 'rsqstat'
 'cemploi' 'zus' 'salaire' 'duree_listes_horsAR' 'lot' 'CLA' 'CVE' 'OPP'
 'Z' 'SAMPLE_CVEOPP' 'femme' 'CS' 'nivetude1' 'nivetude2' 'nivetude3'
 'nivetude4' 'marie' 'motins_lic' 'etranger' 'agegr2635' 'agegr3645'
 'agegr4655' 'agegr56' 'age2gr_2024' 'age2gr_2529' 'age2gr_3034'
 'age2gr_3539' 'age2gr_4044' 'age2gr_4549' 'age2gr_5054' 'age2gr_5559'
 'age2gr_60p' 'age2gr_mis' 'POIDSEMP_Z' 'age29m' 'age3049' 'age50p'
 'EMPLOI_3MOIS' 'AUTRE_3MOIS' 'RADIE_3MOIS' 'EMPLOI_AR110_3MOIS'
 'SUCCES_OPP_3MOIS' 'acceptationOPP_3MOIS' 'acceptationCVE_3MOIS'
 'POIDS_PZ_3MOIS' 'EMPLOI_6MOIS' 'AUTRE_6MOIS' 'RADIE_6MOIS'
 'EMPLOI_AR110_6MOIS' 'SUCCES_OPP_6MOIS' 'acceptationOPP_6MOIS'
 'acceptationCVE_6MOIS' 'POIDS_PZ_6MOIS' 'EMPLOI_9MOIS' 'AUTRE_9MOIS'
 'RADIE_9MOIS' 'EMPLOI_AR110_9MOIS' 'SUCCES_OPP_9MOIS'
 'acceptationOPP_9MOIS' 'acceptation

In [7]:
df.head()

,ale,mois_saisie_occ,acceptationCVE,acceptationOPP,nregion,ndem,sexe,nenf,nation,motins,...,acceptationCVE_9MOIS,POIDS_PZ_9MOIS,EMPLOI_12MOIS,AUTRE_12MOIS,RADIE_12MOIS,EMPLOI_AR110_12MOIS,SUCCES_OPP_12MOIS,acceptationOPP_12MOIS,acceptationCVE_12MOIS,POIDS_PZ_12MOIS
0,91043,11.0,0.0,0.0,116,2,1,3,01,G,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,91008,12.0,0.0,1.0,116,3,1,0,22,8,...,0.0,0.969888,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.969888
2,91146,5.0,0.0,0.0,116,1,1,2,01,2,...,0.0,0.964660,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.964660
3,91008,6.0,0.0,1.0,116,3,2,2,01,8,...,0.0,0.964660,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.964660
4,91146,3.0,0.0,0.0,116,1,1,1,01,2,...,0.0,1.116305,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.116305


In [8]:
df["rsqstat"].unique()

array(['RS2', 'RS3', '', 'RS1'], dtype=object)

## Housekeeping

In [9]:
# Check monotonicity
print((df["acceptationCVE_6MOIS"] > df["acceptationCVE_9MOIS"]).sum())
print((df["acceptationCVE_9MOIS"] > df["acceptationCVE_12MOIS"]).sum())

df[[
    "acceptationCVE",
    "acceptationCVE_3MOIS",
    "acceptationCVE_6MOIS",
    "acceptationCVE_9MOIS",
    "acceptationCVE_12MOIS"
]].corr()


0
0


,acceptationCVE,acceptationCVE_3MOIS,acceptationCVE_6MOIS,acceptationCVE_9MOIS,acceptationCVE_12MOIS
acceptationCVE,1.000000,0.806931,0.846256,0.902203,0.929725
acceptationCVE_3MOIS,0.806931,1.000000,0.953532,0.894402,0.867925
acceptationCVE_6MOIS,0.846256,0.953532,1.000000,0.937989,0.910222
acceptationCVE_9MOIS,0.902203,0.894402,0.937989,1.000000,0.970398
acceptationCVE_12MOIS,0.929725,0.867925,0.910222,0.970398,1.000000


### Acceptance at 6 months ⊂ acceptance at 9 months ⊂ acceptance at 12 months. They are cumulative of the same event

In [ ]:

core_vars = [
    # assignation + flag common zone
    "CVE",
    "OPP",
    "SAMPLE_CVEOPP",
    # IV + treatment
    "Z",
    "acceptationCVE",
    "acceptationOPP",

    # Outcome
    "EMPLOI_3MOIS",
    "EMPLOI_6MOIS",
    "EMPLOI_9MOIS",
    "EMPLOI_12MOIS",

    # Controls
    "age","femme","marie","nenf","etranger","zus",
    "duree_listes_horsAR",
    "nivetude1","nivetude2","nivetude3","nivetude4",
    "salaire", "exper", "nregion","CS", "rsqstat","motins","cemploi","temps","ndem","nation","mois_saisie_occ",
    
    #poids
    "POIDS_PZ_3MOIS",
    "POIDS_PZ_6MOIS",
    "POIDS_PZ_9MOIS",
    "POIDS_PZ_12MOIS"
    
]

df_eda = df[core_vars].copy()

num_cols = [
    "Z","acceptationCVE","acceptationOPP",
    "EMPLOI_3MOIS","EMPLOI_6MOIS","EMPLOI_9MOIS","EMPLOI_12MOIS",
    "age","femme","marie","nenf","etranger",
    "duree_listes_horsAR",
    "nivetude1","nivetude2","nivetude3","nivetude4"
]

# convert ONLY those to numeric
df_eda[num_cols] = df_eda[num_cols].apply(pd.to_numeric, errors="coerce")
print(df_eda.shape)

KeyError: "['POIDS_PZ_dMOIS'] not in index"

In [28]:
df_eda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43977 entries, 0 to 43976
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   Z                    43977 non-null  float32 
 1   acceptationCVE       43977 non-null  float64 
 2   acceptationOPP       43977 non-null  float32 
 3   EMPLOI_3MOIS         43977 non-null  float32 
 4   EMPLOI_6MOIS         43977 non-null  float32 
 5   EMPLOI_9MOIS         43977 non-null  float32 
 6   EMPLOI_12MOIS        43977 non-null  float32 
 7   age                  43977 non-null  int16   
 8   femme                43977 non-null  float32 
 9   marie                43977 non-null  float32 
 10  nenf                 43977 non-null  int8    
 11  etranger             43977 non-null  float32 
 12  zus                  43977 non-null  object  
 13  duree_listes_horsAR  43977 non-null  float32 
 14  nivetude1            43977 non-null  float32 
 15  nivetude2          

## PART 1 : Baseline Imbalance and Selection into Treatment
### Treatment was not randomized (Assignment to treatment was). The goal here is to check whether there are imbalances between participants who accepted treatment and participants who did not (or were not offered) 

In [37]:
df_eda["group"] = np.select(
    [
        df_eda["acceptationCVE"] == 1,
        df_eda["acceptationOPP"] == 1
    ],
    ["CVE", "OPP"],
    default="non_traite"
)

df_eda["group"].value_counts()

group
non_traite    26620
OPP           15687
CVE            1670
Name: count, dtype: int64

In [38]:
df_eda.to_csv("data/df_eda_clean.csv", index=False)

## 1- t-test on the difference in means

In [14]:
from scipy.stats import ttest_ind

results = []

X_cols = [
    "age","femme","marie","nenf","etranger",
    "duree_listes_horsAR",
    "nivetude1","nivetude2","nivetude3","nivetude4"
]



for var in X_cols:
    
    # CVE vs non_traite
    x_cve = df_eda.loc[df_eda["group"]=="CVE", var]
    x_nt  = df_eda.loc[df_eda["group"]=="non_traite", var]
    
    t_cve, p_cve = ttest_ind(x_cve, x_nt, equal_var=False)
    
    # OPP vs non_traite
    x_opp = df_eda.loc[df_eda["group"]=="OPP", var]
    t_opp, p_opp = ttest_ind(x_opp, x_nt, equal_var=False)
    
    results.append([
        var,
        x_cve.mean(),
        x_opp.mean(),
        x_nt.mean(),
        p_cve,
        p_opp
    ])

balance_test = pd.DataFrame(results, columns=[
    "variable",
    "mean_CVE",
    "mean_OPP",
    "mean_non_traite",
    "p_CVE_vs_non_traite",
    "p_OPP_vs_non_traite"
])

balance_test.sort_values("p_CVE_vs_non_traite")

,variable,mean_CVE,mean_OPP,mean_non_traite,p_CVE_vs_non_traite,p_OPP_vs_non_traite
5,duree_listes_horsAR,237.972809,236.822372,215.615784,1.019536e-14,5.711753e-71
4,etranger,0.836527,0.829987,0.802292,2.674777e-04,8.129997e-13
0,age,37.179641,37.163957,36.421187,3.160788e-03,1.635553e-12
9,nivetude4,0.173653,0.178810,0.195905,2.037504e-02,1.226880e-05
8,nivetude3,0.318563,0.304838,0.292261,2.519150e-02,6.406653e-03
2,marie,0.467665,0.477338,0.452817,2.383795e-01,1.048048e-06
6,nivetude1,0.307186,0.320329,0.316566,4.206545e-01,4.225504e-01
3,nenf,0.905389,0.930516,0.885387,5.220870e-01,3.077034e-04
7,nivetude2,0.200599,0.196022,0.195079,5.847136e-01,8.132622e-01
1,femme,0.499401,0.504877,0.498610,9.500080e-01,2.130754e-01


### First, participants in both CVE and OPP exhibit a substantially longer prior unemployment duration (duree_listes_horsAR)compared to non-treated individuals (approximately +22 days). This suggests that treated individuals tend to be more disadvantaged exante.
### Second, treated individuals are more likely to be foreign-born (etranger) and are slightly older on average. While the age difference is statistically significant, its magnitude remains economically small (less than one year on average).
### Differences also appear across education categories. Participants are relatively less represented in the highest education group nivetude4) and slightly more represented in intermediate education levels, indicating compositional differences in skill profiles across groups.
### In contrast, some characteristics such as gender (femme) show virtually no difference across groups, suggesting balance along this dimension

## 2- Standardized Mean Difference

In [15]:
import numpy as np
import pandas as pd

# Fonction SMD
def smd(x1, x2):
    m1, m2 = x1.mean(), x2.mean()
    v1, v2 = x1.var(ddof=1), x2.var(ddof=1)
    return (m1 - m2) / np.sqrt((v1 + v2) / 2)

results = []

for var in X_cols:
    
    x_cve = df_eda.loc[df_eda["group"]=="CVE", var]
    x_opp = df_eda.loc[df_eda["group"]=="OPP", var]
    x_nt  = df_eda.loc[df_eda["group"]=="non_traite", var]
    
    results.append([
        var,
        smd(x_cve, x_nt),
        smd(x_opp, x_nt),
        smd(x_cve, x_opp)
    ])

smd_table = pd.DataFrame(results, columns=[
    "variable",
    "SMD_CVE_vs_non_traite",
    "SMD_OPP_vs_non_traite",
    "SMD_CVE_vs_OPP"
])

# Trier par déséquilibre CVE
smd_table.sort_values("SMD_CVE_vs_non_traite", key=lambda s: abs(s), ascending=False)

,variable,SMD_CVE_vs_non_traite,SMD_OPP_vs_non_traite,SMD_CVE_vs_OPP
5,duree_listes_horsAR,0.187926,0.177572,0.010145
4,etranger,0.089072,0.071539,0.017544
0,age,0.073081,0.070888,0.001530
9,nivetude4,-0.057350,-0.043820,-0.013535
8,nivetude3,0.057119,0.027485,0.029629
2,marie,0.029787,0.049175,-0.019374
6,nivetude1,-0.020248,0.008076,-0.028323
3,nenf,0.016040,0.036241,-0.020337
7,nivetude2,0.013854,0.002378,0.011476
1,femme,0.001582,0.012533,-0.010949


### The largest imbalance concerns prior unemployment duration (duree_listes_horsAR), with SMD values of approximately 0.18 for both CVE and OPP relative to non-treated individuals. This magnitude is close to the conventional 0.2 threshold and indicates a meaningful difference in baseline labor market attachment.

### Other covariates display smaller imbalances. Migration status (etranger) and age show SMD values below 0.1, suggesting relatively mild differences across groups. Education categories exhibit modest deviations, particularly for the highest education group (nivetude4), though these remain well below 0.1 in absolute value.

### Importantly, gender (femme) appears almost perfectly balanced across groups.

## 3- Omnibus balance test
### Instead of testing each variable separately, we can test whether all baseline covariates jointly predict treatment status.

In [16]:
# 0) Make sure X are numeric
df_eda[X_cols] = df_eda[X_cols].apply(pd.to_numeric, errors="coerce")

def omnibus_balance_test(df, dep_col, X_cols, subset_mask=None):
    """
    Runs: dep = alpha + X'beta + e
    Returns the joint F-test of H0: beta = 0 (all covariates jointly zero).
    """
    d = df.copy()
    if subset_mask is not None:
        d = d.loc[subset_mask].copy()

    # Keep only needed columns and drop missing
    d = d[[dep_col] + X_cols].dropna()

    y = pd.to_numeric(d[dep_col], errors="coerce")
    X = sm.add_constant(d[X_cols])

    model = sm.OLS(y, X).fit()

    # Joint test H0: all slopes = 0
    test = model.f_test(np.eye(len(X_cols), len(X_cols)+1, k=1))  # excludes constant
    return model, test

# ============================================================
# A) Omnibus balance test for TAKE-UP (treatment status)
# CVE take-up vs non-treated 
mask_cve = df_eda["group"].isin(["CVE", "non_traite"])
df_eda.loc[mask_cve, "D_cve"] = (df_eda.loc[mask_cve, "group"] == "CVE").astype(int)

model_cve, test_cve = omnibus_balance_test(df_eda, "D_cve", X_cols, subset_mask=mask_cve)
print("Omnibus balance test (CVE vs non_traite):")
print(test_cve)

# OPP take-up vs non-treated 
mask_opp = df_eda["group"].isin(["OPP", "non_traite"])
df_eda.loc[mask_opp, "D_opp"] = (df_eda.loc[mask_opp, "group"] == "OPP").astype(int)

model_opp, test_opp = omnibus_balance_test(df_eda, "D_opp", X_cols, subset_mask=mask_opp)
print("\nOmnibus balance test (OPP vs non_traite):")
print(test_opp)

Omnibus balance test (CVE vs non_traite):
<F test: F=8.068794140775157, p=3.8561043989913676e-13, df_denom=2.83e+04, df_num=10>

Omnibus balance test (OPP vs non_traite):
<F test: F=44.27727010232646, p=2.2043796097971388e-88, df_denom=4.23e+04, df_num=10>


### The omnibus F-tests strongly reject the null hypothesis that baseline covariates jointly do not predict treatment participation. This confirms the presence of systematic selection into both CVE and OPP programs. In particular, the imbalance appears substantially stronger for OPP participant

### Conclusion : Treatment participation is clearly non-random with respect to observable characteristics. Participants differ systematically from non-participants, particularly in prior unemployment duration. These findings indicate the presence of selection on observables and confirm that naïve comparisons of outcomes would be biased. This justifies the use of a causal identification strategy based on randomized assignment and appropriate adjustment for baseline covariates

## Heterogeneity by Ex-Ante Risk and the Parking Hypothesis

### This exploratory section examines whether employment differences between CVE and OPP vary across levels of predicted risk of long-term unemployment (rsqstat).
### If private providers respond to performance-based incentives by prioritizing easier-to-place individuals, the employment gap should be larger among high-risk job seekers.
### While purely descriptive, this analysis provides preliminary evidence on heterogeneous patterns and motivates a formal heterogeneous treatment effect estimation in the next steps.

In [17]:
df_eda.head()

,Z,acceptationCVE,acceptationOPP,EMPLOI_3MOIS,EMPLOI_6MOIS,EMPLOI_9MOIS,EMPLOI_12MOIS,age,femme,marie,...,nivetude3,nivetude4,salaire,exper,nregion,CS,rsqstat,group,D_cve,D_opp
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54,0.0,1.0,...,1.0,0.0,A,34,116,Employé qualifié,RS2,non_traite,0.0,0.0
1,2.0,0.0,1.0,0.0,0.0,0.0,0.0,48,0.0,0.0,...,0.0,1.0,B,10,116,Ouvrier qualifié,RS3,OPP,NaN,1.0
2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,48,0.0,1.0,...,1.0,0.0,C,05,116,Employé qualifié,RS3,non_traite,0.0,0.0
3,2.0,0.0,1.0,0.0,0.0,0.0,0.0,42,1.0,0.0,...,1.0,0.0,D,25,116,Employé qualifié,,OPP,NaN,1.0
4,2.0,0.0,0.0,0.0,0.0,0.0,0.0,40,0.0,0.0,...,1.0,0.0,D,15,116,Employé qualifié,RS3,non_traite,0.0,0.0


In [18]:
df_eda["group"].unique()

array(['non_traite', 'OPP', 'CVE'], dtype=object)

In [19]:
import numpy as np

df_h = df_eda[df_eda["group"].isin(["CVE","OPP"])].copy()

df_h["rsqstat"] = df_h["rsqstat"].replace("", np.nan)
df_h = df_h.dropna(subset=["rsqstat", "EMPLOI_6MOIS"])

rates = df_h.groupby(["rsqstat","group"])["EMPLOI_6MOIS"].mean().unstack()

print("Columns in rates:", rates.columns)   # <- tu verras si OPP existe
rates["gap_OPP_minus_CVE"] = rates.get("OPP") - rates.get("CVE")  # <- ne plante jamais

rates

Columns in rates: Index(['CVE', 'OPP'], dtype='object', name='group')


group,CVE,OPP,gap_OPP_minus_CVE
rsqstat,,,
RS1,0.437500,0.296236,-0.141264
RS2,0.235725,0.205367,-0.030358
RS3,0.184932,0.138140,-0.046791


In [20]:
import numpy as np
import pandas as pd

df_h = df_eda[df_eda["group"].isin(["CVE","OPP"])].copy()
print("After keeping CVE/OPP:", df_h.shape)
print(df_h["group"].value_counts())

# rsqstat empty strings -> NaN
df_h["rsqstat"] = df_h["rsqstat"].replace("", np.nan)
print("\nMissing rsqstat:", df_h["rsqstat"].isna().mean())

# check EMPLOI_6MOIS type / missing
print("EMPLOI_6MOIS dtype:", df_h["EMPLOI_6MOIS"].dtype)
print("Missing EMPLOI_6MOIS:", df_h["EMPLOI_6MOIS"].isna().mean())

# dropna
df_h2 = df_h.dropna(subset=["rsqstat", "EMPLOI_6MOIS"])
print("\nAfter dropna(rsqstat, EMPLOI_6MOIS):", df_h2.shape)
print(df_h2["group"].value_counts())

# if not empty, build rates
if len(df_h2) > 0:
    rates = df_h2.groupby(["rsqstat","group"])["EMPLOI_6MOIS"].mean().unstack()
    print("\nRates columns:", rates.columns)
    print(rates)

After keeping CVE/OPP: (17357, 26)
group
OPP    15687
CVE     1670
Name: count, dtype: int64

Missing rsqstat: 0.15964740450538686
EMPLOI_6MOIS dtype: float32
Missing EMPLOI_6MOIS: 0.0

After dropna(rsqstat, EMPLOI_6MOIS): (14586, 26)
group
OPP    13255
CVE     1331
Name: count, dtype: int64

Rates columns: Index(['CVE', 'OPP'], dtype='object', name='group')
group         CVE       OPP
rsqstat                    
RS1      0.437500  0.296236
RS2      0.235725  0.205367
RS3      0.184932  0.138140


### The employment gap between OPP and CVE is particularly large for the most fragile individuals (RS1), reaching −14 percentage points. In contrast, the gap is substantially smaller for RS2 and RS3. This pattern suggests that performance differences across providers are concentrated among the most disadvantaged job seekers, which is consistent with the “parking” hypothesis.

In [21]:
cve = [0.4375, 0.2357, 0.1849]
opp = [0.2962, 0.2054, 0.1381]

x = range(len(rsq))
width = 0.35

plt.figure(figsize=(7,5))
plt.bar([i - width/2 for i in x], cve, width=width, label="CVE")
plt.bar([i + width/2 for i in x], opp, width=width, label="OPP")

plt.xticks(x, rsq)
plt.title("Employment Gap (OPP − CVE) by Predicted Risk of Long-Term Unemployment")
plt.xlabel("Risk of Long-Term Unemployment")
plt.ylabel("Employment Rate (6 months)")
plt.legend()
plt.show()

NameError: name 'rsq' is not defined